<b>Talk to Gemini with local speech-to-text</b>

Have a spoken conversation with Gemini without sending your audio to a cloud speech API. This version records audio in the notebook and runs transcription locally before forwarding the text to Gemini.


In [1]:
#@title Install local speech processing dependencies

!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip install -q openai-whisper ipywebrtc soundfile


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.0/305.0 kB 3.4 MB/s eta 0:00:00


<b>No Google Cloud account required</b>

The notebook now uses an on-device Whisper model for speech recognition, so you can skip any Cloud project setup or billing configuration.


In [ ]:
#@title Configure local transcription defaults

language_code = "en"  # @param ["en", "ko", "ja", "zh", "fr", "de", "es"]
whisper_model_size = "base"  # @param ["tiny", "base", "small", "medium"]

import whisper

try:
    whisper_model  # type: ignore[name-defined]
    if getattr(whisper_model, "_model_name", None) != whisper_model_size:
        whisper_model = whisper.load_model(whisper_model_size)
except NameError:
    whisper_model = whisper.load_model(whisper_model_size)

whisper_model._model_name = whisper_model_size


In [ ]:
# Local transcription replaces the previous Cloud enablement step.
# No additional commands are required here.


In [ ]:
#@title Configure Gemini API key

#Access your Gemini API key

import google.generativeai as genai
from google.colab import userdata

gemini_api_secret_name = 'GOOGLE_API_KEY'  # @param {type: "string"}

try:
  GOOGLE_API_KEY=userdata.get(gemini_api_secret_name)
  genai.configure(api_key=GOOGLE_API_KEY)
except userdata.SecretNotFoundError as e:
   print(f'Secret not found\n\nThis expects you to create a secret named {gemini_api_secret_name} in Colab\n\nVisit https://makersuite.google.com/app/apikey to create an API key\n\nStore that in the secrets section on the left side of the notebook (key icon)\n\nName the secret {gemini_api_secret_name}')
   raise e
except userdata.NotebookAccessError as e:
  print(f'You need to grant this notebook access to the {gemini_api_secret_name} secret in order for the notebook to access Gemini on your behalf.')
  raise e
except Exception as e:
  # unknown error
  print(f"There was an unknown error. Ensure you have a secret {gemini_api_secret_name} stored in Colab and it's a valid key from https://makersuite.google.com/app/apikey")
  raise e

model = genai.GenerativeModel('gemini-pro')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.9/146.9 kB 2.4 MB/s eta 0:00:00


In [ ]:
#@title Setup helpers

import io
import tempfile
from pathlib import Path
import textwrap

from ipywebrtc import AudioRecorder, CameraStream
from IPython.display import Markdown
from google.colab import output

output.enable_custom_widget_manager()

def transcribe_audio_bytes(audio_bytes: bytes) -> str:
    temp_file = Path(tempfile.mkstemp(suffix=".webm")[1])
    try:
        temp_file.write_bytes(audio_bytes)
        result = whisper_model.transcribe(str(temp_file), language=language_code or None)
        return result.get("text", "").strip()
    finally:
        temp_file.unlink(missing_ok=True)

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 4.4 MB/s eta 0:00:00


In [ ]:
#@title Record your speech

# create a microphone stream
camera = CameraStream(constraints={'audio': True, 'video':False})

# create an audio recorder that uses the microphone stream
recorder = AudioRecorder(stream=camera)

# display the recorder widget
recorder


AudioRecorder(audio=Audio(value=b'', format='webm'), stream=CameraStream(constraints={'audio': True, 'video': …

In [ ]:
#@title Transcribe and send to Gemini

recorded_audio = recorder.audio.value

if recorded_audio is None:
    raise ValueError("Please record audio before running the transcription cell.")

audio_text = transcribe_audio_bytes(recorded_audio)

if not audio_text:
    raise ValueError("No speech was detected in the recording. Try again with clearer audio.")

response = model.generate_content(audio_text)

to_markdown(f'**You**: {audio_text}

**Gemini**:
{response.text}')


> **You**: Can you compose a sketch for Saturday Night Live that includes corgis and Keanu Reeves?
> 
> **Gemini**:
> Title: Keanu Reeves and the Corgi Kingdom
> 
> [Scene: A magical forest. Keanu Reeves is walking through the forest, dressed in a wizard's robe.]
> 
> Keanu Reeves: (to himself) I am Keanu Reeves, the Great Wizard of Corgis. I must find the lost kingdom of the corgis.
> 
> [Keanu continues walking and comes across a group of corgis playing in a clearing.]
> 
> Keanu Reeves: (excited) Corgis!
> 
> [The corgis stop playing and look at Keanu.]
> 
> Keanu Reeves: I am here to help you. I will lead you to your lost kingdom.
> 
> [The corgis bark happily and start following Keanu.]
> 
> [Keanu and the corgis walk through the forest, encountering various obstacles along the way. They are attacked by a pack of wolves, but Keanu uses his magic to defeat them.]
> 
> [Finally, they reach the lost kingdom of the corgis. The corgis are overjoyed and celebrate Keanu's arrival.]
> 
> Corgi King: (bowing to Keanu) Thank you, Great Wizard of Corgis. You have saved our kingdom.
> 
> Keanu Reeves: (smiling) You're welcome, Corgi King. I am glad I could help.
> 
> [Keanu and the corgis live happily ever after in the lost kingdom.]
> 
> [End Scene]